### Notebook : 02_shared_state_and_helpers

##### 1. Purpose

- This notebook defines the shared workflow state and reusable state-management functions.

- It contains:

    - MultiAgentState
    - create_initial_state()
    - find_assigned_agent_task()
    - validate_task_dependencies()
    - store_agent_result()
    - record_agent_execution()
    - record_agent_error()

- It does not execute any specialist tools or agents.

##### 2. Technologies Used

- Python
- TypedDict
- Pydantic models from 01_shared_models
- Type hints
- Reusable helper functions
- Databricks %run

##### 3. Input

This notebook supports the following inputs:

- Customer request
- Coordinator execution-plan tasks
- Agent execution status and messages
- Workflow error details

##### 4. Output

This notebook produces:

- Initialized MultiAgentState
- Assigned AgentTask or None
- Validated ExecutionRecord entries
- Standardized workflow-error entries

##### 5. Architecture

``` text

Customer request
       │
       ▼
create_initial_state()
       │
       ▼
MultiAgentState
       │
       ├── user_request
       ├── coordinator_result
       ├── agent_results
       ├── execution_history
       ├── final_response
       └── errors
               │
               ▼
      Reusable helper functions

```

##### 6. Load shared schemas

In [0]:
%run ./01_shared_models

##### 7. Additional imports

In [0]:
from typing import Any, Dict, List, Optional, TypedDict

##### 8. Shared-state definition

In [0]:
class MultiAgentState(TypedDict):
    """
    Shared state passed between components in the
    multi-agent workflow.
    """

    user_request: str

    coordinator_result: Optional[
        CoordinatorResult
    ]

    agent_results: Dict[
        AgentName,
        BaseAgentResult,
    ]

    execution_history: List[
        ExecutionRecord
    ]

    final_response: Optional[str]

    errors: List[
        AgentErrorRecord
    ]

##### 9. Initial-state function

In [0]:
def create_initial_state(
    user_request: str,
) -> MultiAgentState:
    """
    Create the initial shared state for one customer request.

    Parameters
    ----------
    user_request:
        Original request submitted by the customer.

    Returns
    -------
    MultiAgentState
        New shared state containing empty workflow results.

    Raises
    ------
    ValueError
        If the request is empty or contains only whitespace.
    """

    cleaned_request = user_request.strip()

    if not cleaned_request:
        raise ValueError(
            "user_request must contain at least one non-whitespace character."
        )

    return {
        "user_request": cleaned_request,
        "coordinator_result": None,
        "agent_results": {},
        "execution_history": [],
        "final_response": None,
        "errors": [],
    }

##### 10. Helper: validate task dependencies

In [0]:
def validate_task_dependencies(
    state: MultiAgentState,
    task: AgentTask,
) -> None:
    """
    Verify that all dependencies required by a task exist
    and completed successfully.

    Raises
    ------
    ValueError
        When a dependency is missing or unsuccessful.
    """

    agent_results = state.get(
        "agent_results",
        {},
    )

    for dependency_name in task.depends_on:
        dependency_result = (
            agent_results.get(
                dependency_name
            )
        )

        if dependency_result is None:
            raise ValueError(
                f"Task {task.task_id!r} cannot run "
                f"because dependency "
                f"{dependency_name!r} is missing."
            )

        if dependency_result.status != "success":
            raise ValueError(
                f"Task {task.task_id!r} cannot run "
                f"because dependency "
                f"{dependency_name!r} has status "
                f"{dependency_result.status!r}."
            )

##### 11. Helper: Find an assigned task

In [0]:
def find_assigned_agent_task(
    coordinator_result: CoordinatorResult,
    agent_name: AgentName,
) -> Optional[AgentTask]:
    """
    Return the Coordinator task assigned to one agent.

    The current workflow supports at most one task per agent.
    """

    for task in coordinator_result.execution_plan:
        if task.agent_name == agent_name:
            return task

    return None

##### 12. Helper: Add Agent result to the shared state

In [0]:
def store_agent_result(
    state: MultiAgentState,
    agent_result: BaseAgentResult,
) -> None:
    """
    Store an agent result in the shared state.
    """

    state["agent_results"][
        agent_result.agent_name
    ] = agent_result

##### 13. Helper: Record Agent Execution

In [0]:
def record_agent_execution(
    state: MultiAgentState,
    agent_name: WorkflowComponent,
    status: AgentStatus,
    message: str,
) -> None:
    """
    Add one validated execution event to shared state.

    This function mutates the shared-state dictionary directly.
    Therefore, it does not return the state.
    """

    execution_record = ExecutionRecord(
        agent_name=agent_name,
        status=status,
        message=message,
    )

    state["execution_history"].append(
        execution_record
    )

##### 14. Helper: Record Agent or Workflow Error

In [0]:
def record_agent_error(
    state: MultiAgentState,
    agent_name: WorkflowComponent,
    error_code: str,
    error_message: str,
) -> None:
    """
    Add one validated agent or workflow error to shared state.

    This function mutates the shared-state dictionary directly.
    Therefore, it does not return the state.
    """

    error_record = AgentErrorRecord(
        agent_name=agent_name,
        error_code=error_code,
        error_message=error_message,
    )

    state["errors"].append(
        error_record
    )

##### 15. Independent Test Function

In [0]:
def test_shared_state_and_helpers() -> None:
    """
    Run independent validation tests for shared-state
    initialization and reusable helper functions.
    """

    test_state = create_initial_state(
        "Video streaming keeps buffering."
    )

  
    test_tasks = [
        AgentTask(
            task_id="task_1",
            agent_name=PREDICTION_AGENT_NAME,
            task_description=(
                "Predict customer churn."
            ),
        ),
        AgentTask(
            task_id="task_2",
            agent_name=FINAL_RESPONSE_AGENT_NAME,
            task_description=(
                "Generate the final customer-facing "
                "response."
            ),
            depends_on=[
                PREDICTION_AGENT_NAME,
            ],
        ),
    ]

    test_coordinator_result = CoordinatorResult(
        status="success",
        message=(
            "Execution plan created successfully."
        ),
        request_type="prediction",
        reasoning=(
            "The request requires a churn prediction."
        ),
        execution_plan=test_tasks,
    )

    assigned_task = find_assigned_agent_task(
        coordinator_result=test_coordinator_result,
        agent_name=PREDICTION_AGENT_NAME,
    )

    missing_task = find_assigned_agent_task(
        coordinator_result=test_coordinator_result,
        agent_name=RETENTION_AGENT_NAME,
    )

    record_agent_execution(
        state=test_state,
        agent_name="prediction_agent",
        status="success",
        message="Prediction task completed successfully.",
    )

    record_agent_error(
        state=test_state,
        agent_name="prediction_agent",
        error_code="TEST_ERROR",
        error_message=(
            "Example error used to validate error recording."
        ),
    )

    record_agent_execution(
        state=test_state,
        agent_name=ORCHESTRATOR_NAME,
        status="running",
        message="Workflow execution started.",
    )

    record_agent_error(
        state=test_state,
        agent_name=ORCHESTRATOR_NAME,
        error_code="TEST_WORKFLOW_ERROR",
        error_message="Example orchestration error.",
    )

    assert (
        test_state["user_request"]
        == "Video streaming keeps buffering."
    )

    assert (
        test_state["coordinator_result"]
        is None
    )

    assert (
        test_state["agent_results"]
        == {}
    )

    assert assigned_task is not None

    assert (
        assigned_task.agent_name
        == PREDICTION_AGENT_NAME
    )

    assert missing_task is None

    assert len(
        test_state["execution_history"]
    ) == 2

    assert (
        test_state["execution_history"][0].agent_name
        == PREDICTION_AGENT_NAME
    )

    assert (
        test_state["execution_history"][0].status
        == "success"
    )

    assert len(
        test_state["errors"]
    ) == 2

    assert (
        test_state["errors"][0].error_code
        == "TEST_ERROR"
    )

    assert (
    test_state["execution_history"][-1].agent_name
    == ORCHESTRATOR_NAME
    )

    assert (
    test_state["errors"][-1].agent_name
    == ORCHESTRATOR_NAME
    )

    try:
        create_initial_state("   ")

    except ValueError:
        pass

    else:
        raise AssertionError(
            "An empty request should raise ValueError."
        )

    print(
        "All shared-state and helper tests passed."
    )


##### 16. Notes

- Agent notebooks contain execution logic.
- Shared state contains communication between agents.
- Shared models define the communication contracts.


##### 17. Key Learnings

- TypedDict documents the complete structure of the shared workflow state.

- Shared state allows agents to exchange structured information without relying on unstructured conversation history.

- create_initial_state() ensures that every customer request begins with an isolated and valid state object.

- Reusable helper functions prevent duplicated state-management logic across agent notebooks.

- get_agent_task() separates execution-plan lookup from specialist-agent logic.

- The current design assumes that each specialist agent receives at most one task.

- record_agent_execution() creates consistent and validated workflow-history records.

- record_agent_error() creates consistent and validated error records.

- Helper functions can update mutable state directly and therefore do not need to return it.

- Structured agent results remain separate from the final customer-facing response.

- Consistent status, history, and error records make the workflow easier to test and debug.

##### 18. Conclusion

- In this notebook, we created the shared state used by the multi-agent workflow and implemented reusable functions for initializing state, locating assigned tasks, recording execution history, and recording agent or workflow errors.

- Centralizing this behavior prevents the Coordinator, SQL, Prediction, Vector Search, Retention, and Final Response notebooks from duplicating state-management logic. Each downstream agent notebook can now focus only on its unique responsibility while exchanging information through the same structured workflow state.

##### 19. Next Notebook

03_coordinator_agent

The Coordinator Agent will:

- Read the customer request.
- Identify the request type.
- Select the required specialist agents.
- Create an ordered execution plan.
- Define task dependencies.
- Return a validated CoordinatorResult.
- Store the Coordinator result in shared state.